# RL-07b — Sur-généralisation relative dans le Climbing Game

**Série** : Reinforcement Learning | **Niveau** : intermédiaire | **Durée estimée** : 45 min

**Navigation** : [← RL-07 Multi-Agent](RL-07-Apprentissage-Multi-Agent-Python.ipynb) | **RL-07b Climbing Game** | [Python](RL-07b-Sur-Generalisation-Climbing-Game-Python.ipynb)

## Objectifs d’apprentissage
- reconnaître la **sur-généralisation relative** dans un jeu coopératif non dégénéré ;
- implémenter IQL et sa variante **hysteretic** ;
- comparer plusieurs graines avec IQM et bootstrap ;
- distinguer non-stationnarité, exploration et biais de mauvaise coordination.

**Prérequis** : Q-learning tabulaire, politique $\varepsilon$-greedy, équilibres de Nash. Toutes les expériences sont CPU et déterministes par graine.

## Question directrice

Dans un problème mono-agent, une action possède une distribution de récompenses fixée par l'environnement. Ici, la récompense associée à `A` dépend aussi de l'action choisie au même instant par l'autre apprenant. L'expérience cherchera donc moins à répondre « quelle méthode obtient la meilleure courbe ? » qu'à établir la chaîne causale suivante : exploration indépendante → mauvaises coordinations transitoires → estimation marginale pessimiste de `A` → destination empirique sous-optimale au budget fini.

Le notebook suit le *Climbing Game* introduit comme banc d'essai de l'apprentissage coopératif indépendant par Claus et Boutilier (1998). La mise à jour asymétrique étudiée ensuite correspond au principe d'hysteretic Q-learning de Matignon, Laurent et Le Fort-Piat (2007). Ces références motivent le mécanisme ; les nombres affichés ici proviennent exclusivement de l'exécution locale multi-graines.


## 1. Un jeu coopératif qui piège les apprenants indépendants

| Agent 1 \ Agent 2 | A | B | C |
|---|---:|---:|---:|
| **A** | **11** | -30 | 0 |
| **B** | -30 | 7 | 6 |
| **C** | 0 | 0 | 5 |

L’optimum conjoint est `(A,A)` avec 11. Mais une exploration non coordonnée de `A` rencontre souvent -30. Chaque agent peut sous-évaluer `A` et se réfugier dans `(C,C)` : la qualité d’une action est diluée par les actions simultanées de l’autre agent.

### Trois propriétés à ne pas confondre

1. `(A,A)` est le **maximum global** de la matrice, mais `A` n'est pas individuellement sûre : `(A,B)` et `(B,A)` valent -30.
2. Les seuls Nash purs sont `(A,A)` et `(B,B)`. `(C,C)` n’est pas Nash : l’agent de ligne dévie vers B et gagne 6 au lieu de 5. Une destination modale C/C au budget fini ne prouve donc pas une convergence vers un équilibre.
3. Le jeu est pleinement coopératif : les deux agents reçoivent exactement la même récompense. L'échec ne vient donc ni d'intérêts opposés ni d'un problème d'attribution entre récompenses différentes.

La matrice n’est pas symétrique sous échange des agents, même si la récompense est partagée. Les moyennes de lignes concernent l’agent1 ; les moyennes de colonnes de l’agent2 diffèrent.

La sur-généralisation est dite **relative** parce qu'un agent évalue une action en moyenne relativement à la politique courante de son partenaire. Au début, cette politique explore presque uniformément. Pour `A`, la moyenne uniforme vaut $(11-30+0)/3=-6{,}33$, alors que pour `C` elle vaut $(0+0+5)/3=1{,}67$. Une action indispensable à l'optimum paraît ainsi moins bonne quand on marginalise sur un partenaire encore mal coordonné. Ce calcul ne prédit pas à lui seul la trajectoire stochastique, mais rend le piège falsifiable avant la simulation.


In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Text;
using System.Security;
using Microsoft.DotNet.Interactive.Formatting;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
record FigureSvg(string Text);
Formatter.Register<FigureSvg>((f, w) => w.Write(f.Text), "text/html");
static class Climbing
{
    public static readonly string[] Actions =
    {
        "A",
        "B",
        "C"
    };
    public static readonly double[, ] Payoff =
    {
        {
            11,
            -30,
            0
        },
        {
            -30,
            7,
            6
        },
        {
            0,
            0,
            5
        }
    };
    public const int Episodes = 12000, Tail = 1000;
    public static readonly int[] Seeds = Enumerable.Range(0, 20).ToArray();
    public static long Runs = 0, Interactions = 0;
    public static void Check(bool c, string m)
    {
        if (!c)
            throw new Exception(m);
    }

    public static string Pair(int p) => $"({Actions[p / 3]},{Actions[p % 3]})";
    public static void Matrix(double[, ] m)
    {
        Check(m is not null && m.GetLength(0) == 3 && m.GetLength(1) == 3 && m.Cast<double>().All(double.IsFinite), "matrice3x3finie");
    }

    public static void Rate(double x)
    {
        Check(double.IsFinite(x) && x >= 0 && x <= 1, "taux dans[0,1]");
    }
}

var payoff = Climbing.Payoff;
var nash = (
    from a in Enumerable.Range(0, 3)from b in Enumerable.Range(0, 3)
    where payoff[a, b] == Enumerable.Range(0, 3).Max(i => payoff[i, b]) && payoff[a, b] == Enumerable.Range(0, 3).Max(j => payoff[a, j])select a * 3 + b).ToArray();
Console.WriteLine($".NET{Environment.Version}, matrice3x3, optimum(A,A)={payoff[0, 0]}");
Console.WriteLine("Nash purs : " + string.Join(", ", nash.Select(Climbing.Pair)));
Console.WriteLine($"Déviation C→B contre C : {payoff[1, 2]} > {payoff[2, 2]}");
Climbing.Check(nash.SequenceEqual(new[] { 0, 4 }), "Nash exacts");


.NET10.0.14, matrice3x3, optimum(A,A)=11


Nash purs : (A,A), (B,B)


Déviation C→B contre C : 6 > 5


### Lecture du résultat : l'optimum de la matrice (code[0])

Le maximum imprimé confirme que `(A,A)` domine les autres coordinations. Il faut préserver une action excellente malgré ses résultats catastrophiques lorsqu’elle est mal coordonnée.

Le calcul exhaustif de la matrice sert de garde-fou : il sépare la qualité normative de l'optimum de ce que les agents apprendront effectivement. Si IQL présente une autre paire dominante à la fin du budget, on pourra parler d'un défaut d'apprentissage plutôt que d'une ambiguïté sur l'objectif.

La récompense 11 est commune aux deux agents. Les courbes ultérieures montrent donc le retour d'équipe par épisode, sans somme supplémentaire qui doublerait artificiellement l'échelle.


## 2. Même agent tabulaire, deux règles de mise à jour

Chaque agent possède sa propre table Q et choisit par $\varepsilon$-greedy. IQL utilise $\alpha^+=\alpha^-$ ; hysteretic Q-learning utilise $\alpha^-<\alpha^+$. La récompense reste identique : ce n’est pas un bonus ad hoc.

### Équation de mise à jour

Le jeu n'a qu'un état et chaque épisode se termine après une action conjointe. La cible de Q-learning se réduit donc à la récompense immédiate :

$$
\delta_t = r_t-Q_t(a_t), \qquad
Q_{t+1}(a_t)=Q_t(a_t)+\alpha(\delta_t)\,\delta_t.
$$

Pour IQL, $\alpha(\delta)=0{,}1$ quel que soit le signe. Pour la variante hysteretic, $\alpha(\delta)=\alpha^+$ si $\delta\geq0$ et $\alpha^-$ sinon, avec $\alpha^-<\alpha^+$. Un TD-error négatif fait baisser la valeur ; l’hystérésis le fait plus lentement. Le signe ne distingue pas à lui seul une mauvaise coordination, un aléa et une action réellement mauvaise. C'est important : le code ne filtre aucune observation, n'ajoute aucun bonus et ne connaît pas la matrice optimale.

Les tables sont indépendantes et aucune action de l'autre agent n'entre dans l'observation. Du point de vue de chaque table, la dynamique de l'autre politique rend la distribution des récompenses non stationnaire. Le tirage aléatoire entre actions ex æquo évite en outre de favoriser `A` par son indice.


In [2]:
class AgentTabulaire
{
    public double[] Q { get; } = new double[3];
    public double AlphaPlus { get; }
    public double AlphaMoins { get; }

    public AgentTabulaire(double ap, double an)
    {
        Climbing.Rate(ap);
        Climbing.Rate(an);
        AlphaPlus = ap;
        AlphaMoins = an;
    }

    public int Act(Random rng, double epsilon)
    {
        Climbing.Rate(epsilon);
        if (rng.NextDouble() < epsilon)
            return rng.Next(3);
        double max = Q.Max();
        var best = Enumerable.Range(0, 3).Where(i => Q[i] == max).ToArray();
        return best[rng.Next(best.Length)];
    }

    public void Update(int action, double reward)
    {
        Climbing.Check(action >= 0 && action < 3 && double.IsFinite(reward), "update fini et action valide");
        double delta = reward - Q[action];
        Q[action] += (delta >= 0 ? AlphaPlus : AlphaMoins) * delta;
    }
}

Console.WriteLine("Agent tabulaire : choix epsilon-greedy et ex-aequo uniformes, update terminal sans bootstrap.");


Agent tabulaire : choix epsilon-greedy et ex-aequo uniformes, update terminal sans bootstrap.


<a id="exercice-1"></a>

### Exercice 1 — Prédire l’effet de l’hystérésis

Proposez une valeur de `alpha_negative` qui protège `A` sans ignorer toutes les observations négatives. **Indice** : comparez les vitesses d’apprentissage positive et négative.

Une hypothèse complète doit aussi annoncer un risque. Quand $\alpha^-$ devient trop petit, l'agent conserve longtemps une estimation optimiste même si les retours négatifs signalent une action réellement mauvaise. L'ablation finale vérifiera donc une plage de valeurs au lieu de ne montrer que le meilleur réglage.

Pour raisonner avant d'exécuter, comparez notamment `0.05` et `0.01` : les deux satisfont $\alpha^-<\alpha^+$, mais la première valeur ne ralentit les corrections négatives que d'un facteur deux, contre dix pour la seconde.


Partie calculable : `TraceMiseAJour(flux, alphaPlus, alphaMoins)` renvoie la trajectoire d’une seule valeur Q, initialisée à zéro. La garde compare chaque update positif/négatif sur plusieurs flux et taux, y compris alpha−0. Votre hypothèse textuelle est seulement reçue pour discussion ; ce contrôle ne valide pas son explication.


In [3]:
#nullable enable
string? hypotheseAlphaNegative = null; // TODO étudiant : proposition et risque à discuter.
static double[]? TraceMiseAJour(double[] flux, double alphaPlus, double alphaMoins)
{
    // TODO étudiant : trajectoire de Q après chaque récompense, Q initiale zéro.
    return null;
}

bool fini1 = true;
foreach (var(flux, ap, an)in new[]
{
    (new double[] { 10, -10, 3, -7 }, .1, .01),
    (new double[] { -4, 8, -3 }, .2, .05),
    (new double[] { 2, -20, 4, -30 }, .1, 0.0)
}

)
{
    var trace = TraceMiseAJour((double[])flux.Clone(), ap, an);
    if (trace is null)
    {
        fini1 = false;
        break;
    }

    Climbing.Check(trace.Length == flux.Length && trace.All(double.IsFinite), "longueur/finitude trace");
    double q = 0;
    for (int i = 0; i < flux.Length; i++)
    {
        double delta = flux[i] - q;
        q += (delta >= 0 ? ap : an) * delta;
        Climbing.Check(Math.Abs(trace[i] - q) < 1e-10, "update positif/négatif");
    }
}

Console.WriteLine(!fini1 ? "Exercice 1 à compléter : trajectoire Q" : hypotheseAlphaNegative is null ? "Traces validées ; hypothèse à compléter" : "Traces validées ; justification reçue à discuter");


Exercice 1 à compléter : trajectoire Q


## 3. Protocole expérimental multi-graines

Chaque condition emploie 12 000 épisodes, le même calendrier d’exploration et les graines 0 à 19. Les 1 000 derniers épisodes déterminent la paire d’actions empirique dominante.

### Contrôles expérimentaux

- **Même budget** : chaque couple méthode–graine reçoit exactement 12 000 interactions.
- **Même exploration** : $\varepsilon_0=1$, décroissance multiplicative `0.9995`, plancher `0.02`.
- **Mêmes graines** : les identifiants sont `0, …, 19` dans chaque condition ; les choix ex-aequo consomment le RNG selon la trajectoire. Cela ne garantit pas les mêmes tirages exogènes entre politiques.
- **Fenêtre terminale** : les 1 000 dernières actions conjointes servent à classifier la coordination dominante et à calculer le retour final.

Le plancher d'exploration maintient quelques désaccords jusque dans la fenêtre terminale. Le critère n'exige donc pas 100 % d'actions identiques : il retient la paire modale et rapporte séparément son taux de dominance. Cette définition évite de confondre une politique stable avec une trajectoire sans aucun bruit.

Une graine initialise ici le générateur unique qui pilote les explorations et les départages des deux agents. Elle rend chaque trajectoire reproductible, mais vingt graines restent nécessaires pour décrire la fréquence des destinations au budget fixé.


In [4]:
record RunClimbing(int Seed, double[] Rewards, int Dominant, double DominanceRate, double TailReturn);
static class Protocole
{
    public static (int paire, int count) Mode(int[] pairs)
    {
        Climbing.Check(pairs.Length > 0, "mode non vide");
        var counts = new Dictionary<int, (int count, int first)>();
        for (int i = 0; i < pairs.Length; i++)
        {
            int p = pairs[i];
            counts[p] = counts.TryGetValue(p, out var c) ? (c.count + 1, c.first) : (1, i);
        }

        var best = counts.OrderByDescending(c => c.Value.count).ThenBy(c => c.Value.first).First();
        return (best.Key, best.Value.count);
    }

    public static RunClimbing Run(int seed, double alphaMoins, double alphaPlus = .1, double[, ] matrice = null)
    {
        var m = matrice ?? Climbing.Payoff;
        Climbing.Matrix(m);
        var rng = new Random(seed);
        var agents = new[]
        {
            new AgentTabulaire(alphaPlus, alphaMoins),
            new AgentTabulaire(alphaPlus, alphaMoins)
        };
        var rewards = new double[Climbing.Episodes];
        var pairs = new int[Climbing.Episodes];
        double epsilon = 1;
        for (int t = 0; t < Climbing.Episodes; t++)
        {
            int a = agents[0].Act(rng, epsilon), b = agents[1].Act(rng, epsilon);
            double r = m[a, b];
            agents[0].Update(a, r);
            agents[1].Update(b, r);
            rewards[t] = r;
            pairs[t] = 3 * a + b;
            epsilon = Math.Max(.02, epsilon * .9995);
        }

        var mode = Mode(pairs.Skip(Climbing.Episodes - Climbing.Tail).ToArray());
        Climbing.Runs++;
        Climbing.Interactions += Climbing.Episodes;
        return new(seed, rewards, mode.paire, mode.count / (double)Climbing.Tail, rewards.Skip(Climbing.Episodes - Climbing.Tail).Average());
    }

    public static void Destinations(string label, RunClimbing[] runs)
    {
        Console.WriteLine(label + " : " + string.Join(", ", runs.GroupBy(r => r.Dominant).Select(g => $"{Climbing.Pair(g.Key)}:{g.Count()}")));
        foreach (int p in new[]
        {
            0,
            4,
            8
        }

        )
            Console.WriteLine($"Coordination suivie{Climbing.Pair(p)} : {runs.Count(r => r.Dominant == p) / (double)runs.Length:P0}");
        Console.WriteLine($"Dominance moyenne dans les queues : {runs.Average(r => r.DominanceRate):P2}");
    }
}

Console.WriteLine($"Protocole : {Climbing.Seeds.Length} graines×{Climbing.Episodes} épisodes, queue{Climbing.Tail}, epsilon1×.9995/plancher.02.");


Protocole : 20 graines×12000 épisodes, queue1000, epsilon1×.9995/plancher.02.


## 4. Baseline IQL : coordination stable, souvent sous-optimale

La baseline emploie $\alpha^+=\alpha^-=0{,}1$. Chaque table Q voit une distribution de récompenses changeante car l’autre politique apprend simultanément.

### Prédiction puis test

Sous exploration précoce, `A` subit les pénalités -30 contre `B`, tandis que `C` ne subit aucune pénalité négative dans sa ligne. IQL incorpore ces deux types d'observations à la même vitesse. La prédiction réfutable est donc une majorité de graines classées `(C,C)` ou `(B,B)`, et un faible taux de `(A,A)`.

Le diagnostic ne repose pas seulement sur le retour moyen. Le dictionnaire des coordinations suivies montre **où** chaque graine finit ; le taux vers `(A,A)` teste directement la récupération de l'optimum conjoint. Si le taux était élevé malgré une courbe faible, l'explication serait plutôt une exploration terminale coûteuse qu'une destination sous-optimale.


Le terme *coordination terminale* est volontaire : une paire dominante asymétrique comme `(B,C)` décrit un échec de coordination empirique, pas un équilibre de Nash. Les coordinations diagonales suivies sont `(A,A)`, `(B,B)` et `(C,C)` ; seules les deux premières sont Nash. Les autres paires sont reportées sans être requalifiées.

In [5]:
var baseline = Climbing.Seeds.Select(s => Protocole.Run(s, .1)).ToArray();
Protocole.Destinations("IQL", baseline);
Console.WriteLine($"Paire modale(A,A) : {baseline.Count(r => r.Dominant == 0) / (double)baseline.Length:P0}");


IQL : (C,C):19, (B,C):1


Coordination suivie(A,A) : 0 %


Coordination suivie(B,B) : 0 %


Coordination suivie(C,C) : 95 %


Dominance moyenne dans les queues : 87.70 %


Paire modale(A,A) : 0 %


### Interprétation de la baseline

La non-stationnarité explique pourquoi les cibles changent ; l’exploration explique les désaccords ; la **sur-généralisation relative** explique pourquoi les -30 sous mauvaise coordination contaminent la valeur de `A`.

Il faut distinguer trois niveaux causaux. La **non-stationnarité** décrit le signal vu par une table quand l'autre politique change. L'**exploration indépendante** produit concrètement les paires désaccordées. La **sur-généralisation relative** est le biais de décision résultant : la valeur marginale de `A` agrège trop fortement les résultats obtenus avec de mauvaises actions partenaires et devient inférieure à celle d'une coordination plus sûre.

Ce notebook ne prétend pas que toute convergence sous-optimale de MARL relève de ce mécanisme. Ici, la matrice, le partage de la récompense et les destinations finales permettent de l'isoler. Dans un environnement multi-états, l'approximation de fonction, le crédit temporel et l'observabilité partielle ajouteraient d'autres causes.


## 5. Hysteretic Q-learning

Nous fixons $\alpha^+=0{,}1$ et $\alpha^-=0{,}01$. La mise à jour négative reste présente, mais pèse dix fois moins.

### Hypothèse mécaniste

Avec $\alpha^-=0{,}01$, une surprise positive modifie Q dix fois plus vite qu'une surprise négative. Lorsqu'une première coordination `(A,A)` révèle 11, quelques échecs d'exploration ne détruisent plus immédiatement cette information. Les deux agents ont alors davantage de temps pour sélectionner simultanément `A`, ce qui rend les observations positives auto-renforçantes.

La comparaison reste contrôlée : initialisation, politique d'exploration, récompense et nombre d'épisodes sont identiques à IQL. Le résultat attendu est une hausse du nombre de graines dont la paire modale est `(A,A)`, pas simplement une hausse de Q non observable.


In [6]:
var hysteretic = Climbing.Seeds.Select(s => Protocole.Run(s, .01)).ToArray();
Protocole.Destinations("Hysteretic", hysteretic);
Console.WriteLine($"Paire modale(A,A) : {hysteretic.Count(r => r.Dominant == 0) / (double)hysteretic.Length:P0}");


Hysteretic : (A,A):20


Coordination suivie(A,A) : 100 %


Coordination suivie(B,B) : 0 %


Coordination suivie(C,C) : 0 %


Dominance moyenne dans les queues : 97.45 %


Paire modale(A,A) : 100 %


### Lecture du résultat : les taux hysteretic (code[5])

La comparaison des taux indique si réduire l’impact des TD-errors négatives protège l’action prometteuse contre les échecs transitoires. Elle ne prouve pas que tout optimisme fonctionne.

La sortie doit être lue comme un contraste expérimental, pas comme un théorème universel. Si le taux optimal passe de presque zéro à presque un avec les mêmes graines, le changement de sensibilité aux erreurs négatives est associé à un changement des destinations sur cette expérience. La section suivante vérifie que ce contraste apparaît également dans les retours terminaux et quantifie son incertitude.

Un éventuel résidu de récompense sous 11 est compatible avec la politique apprise : le plancher $\varepsilon=0{,}02$ continue à provoquer des explorations et donc quelques pénalités -30. C'est pourquoi l'optimum horizontal des courbes est une référence, pas une valeur que la moyenne glissante doit atteindre exactement.


## 6. Courbes, IQM et intervalle bootstrap

L’IQM moyenne les 50 % centraux des graines. Le bootstrap ré-échantillonne les graines, unités indépendantes de l’expérience.

### Pourquoi deux résumés complémentaires ?

La courbe agrège les trajectoires à chaque épisode et révèle la dynamique d'apprentissage, mais son intervalle interquartile n'est pas un intervalle de confiance. L'IQM (*interquartile mean*) tronque les 25 % de graines les plus basses et les 25 % les plus hautes avant de moyenner : elle est moins dominée par quelques bassins atypiques qu'une moyenne brute, tout en utilisant davantage d'information qu'une médiane.

L'intervalle bootstrap ré-échantillonne les **vingt graines**, car ce sont les réplications indépendantes. Ré-échantillonner les 1 000 épisodes terminaux traiterait à tort des observations autocorrélées d'une même politique comme mille expériences indépendantes. Avec seulement vingt graines, l'intervalle reste descriptif et sa largeur doit être rapportée ; il ne transforme pas l'expérience en preuve asymptotique.

Enfin, l'intervalle tracé entre les quantiles 25 % et 75 % décrit l'hétérogénéité des trajectoires, tandis que l'IC bootstrap porte sur l'IQM des retours terminaux. Les deux bandes répondent donc à des questions différentes.


Convention de l’IQM : tri, puis indices floor(0,25n) à ceil(0,75n) exclus ; pour 20 graines, indices 5 à 14. Aux petites tailles, ce n’est pas une troncature fractionnaire. La moyenne glissante valid de largeur 300 donne 11701 points, alignés aux épisodes 300 à 12000. Chaque bootstrap tire 2000 réplications avec remise des 20 graines (seed 2026) ; ses quantiles sont interpolés linéairement.


In [7]:
static class Stats
{
    public static double Quantile(double[] values, double p)
    {
        Climbing.Check(values.Length > 0 && values.All(double.IsFinite) && double.IsFinite(p) && p >= 0 && p <= 1, "quantile fini");
        var a = values.OrderBy(x => x).ToArray();
        double pos = (a.Length - 1) * p;
        int lo = (int)Math.Floor(pos), hi = (int)Math.Ceiling(pos);
        return a[lo] + (pos - lo) * (a[hi] - a[lo]);
    }

    public static double Iqm(double[] values)
    {
        Climbing.Check(values.Length > 0 && values.All(double.IsFinite), "IQM fini non vide");
        var a = values.OrderBy(x => x).ToArray();
        int lo = (int)Math.Floor(.25 * a.Length), hi = (int)Math.Ceiling(.75 * a.Length);
        return a.Skip(lo).Take(hi - lo).Average();
    }

    public static (double score, double low, double high) Bootstrap(double[] values, int seed = 2026, int nBoot = 2000)
    {
        Climbing.Check(values.Length > 0 && values.All(double.IsFinite) && nBoot > 0, "bootstrap valide");
        var rng = new Random(seed);
        var results = new double[nBoot];
        for (int b = 0; b < nBoot; b++)
        {
            var sample = Enumerable.Range(0, values.Length).Select(_ => values[rng.Next(values.Length)]).ToArray();
            results[b] = Iqm(sample);
        }

        return (Iqm(values), Quantile(results, .025), Quantile(results, .975));
    }

    public static double[] Moving(double[] a, int window = 300)
    {
        Climbing.Check(window > 0 && a.Length >= window && a.All(double.IsFinite), "fenêtre valide");
        var output = new double[a.Length - window + 1];
        double s = a.Take(window).Sum();
        output[0] = s / window;
        for (int i = window; i < a.Length; i++)
        {
            s += a[i] - a[i - window];
            output[i - window + 1] = s / window;
        }

        return output;
    }
}

static FigureSvg Courbes((string label, RunClimbing[] runs, string color)[] series)
{
    const int width = 1000, height = 520;
    const double left = 80, top = 60, pw = 890, ph = 360;
    double X(int i) => left + i * pw / (Climbing.Episodes - 300);
    double Y(double v) => top + (12 - v) * ph / 44;
    string N(double x) => x.ToString("F3", CultureInfo.InvariantCulture);
    string E(string s) => SecurityElement.Escape(s);
    var svg = new StringBuilder($"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'><rect width='100%' height='100%' fill='white'/><g font-family='sans-serif' font-size='13' fill='#222'>");
    svg.Append("<text x='500' y='25' text-anchor='middle'>20 graines : moyenne glissante et intervalle interquartile</text>");
    foreach (int tick in new[]
    {
        -30,
        -20,
        -10,
        0,
        10
    }

    )
    {
        double y = Y(tick);
        svg.Append($"<line x1='{left}' y1='{N(y)}' x2='{left + pw}' y2='{N(y)}' stroke='#ddd'/><text x='70' y='{N(y + 4)}' text-anchor='end'>{tick}</text>");
    }

    foreach (int episode in new[]
    {
        300,
        3000,
        6000,
        9000,
        12000
    }

    )
    {
        double x = X(episode - 300);
        svg.Append($"<line x1='{N(x)}' y1='{top}' x2='{N(x)}' y2='{top + ph}' stroke='#eee'/><text x='{N(x)}' y='445' text-anchor='middle'>{episode}</text>");
    }

    int legend = 0;
    foreach (var(label, runs, color)in series)
    {
        var curves = runs.Select(r => Stats.Moving(r.Rewards)).ToArray();
        int n = curves[0].Length;
        var mean = new double[n];
        var low = new double[n];
        var high = new double[n];
        for (int i = 0; i < n; i++)
        {
            var vals = curves.Select(c => c[i]).ToArray();
            mean[i] = vals.Average();
            low[i] = Stats.Quantile(vals, .25);
            high[i] = Stats.Quantile(vals, .75);
        }

        string Points(double[] values, IEnumerable<int> indices) => string.Join(" ", indices.Select(i => $"{N(X(i))},{N(Y(values[i]))}"));
        svg.Append($"<polygon points='{Points(low, Enumerable.Range(0, n))} {Points(high, Enumerable.Range(0, n).Reverse())}' fill='{color}' fill-opacity='.2'/><polyline points='{Points(mean, Enumerable.Range(0, n))}' fill='none' stroke='{color}' stroke-width='1.7'/>");
        int lx = 170 + legend * 250;
        svg.Append($"<line x1='{lx}' y1='490' x2='{lx + 30}' y2='490' stroke='{color}' stroke-width='3'/><text x='{lx + 40}' y='494'>{E(label)} : moyenne / IQR</text>");
        legend++;
    }

    svg.Append($"<line x1='{left}' y1='{N(Y(11))}' x2='{left + pw}' y2='{N(Y(11))}' stroke='black' stroke-dasharray='5 4'/><text x='745' y='494'>Tirets : optimum11</text><line x1='{left}' y1='{top + ph}' x2='{left + pw}' y2='{top + ph}' stroke='black'/><line x1='{left}' y1='{top}' x2='{left}' y2='{top + ph}' stroke='black'/><text x='500' y='470' text-anchor='middle'>Épisode</text><text transform='translate(22 240) rotate(-90)' text-anchor='middle'>Récompense d’équipe (moyenne sur300épisodes)</text></g></svg>");
    return new(svg.ToString());
}

var comparaison = new[]
{
    ("IQL", baseline, "#d95f02"),
    ("Hysteretic", hysteretic, "#1b9e77")
};
display(Courbes(comparaison));
foreach (var(label, runs, color)in comparaison)
{
    var s = Stats.Bootstrap(runs.Select(r => r.TailReturn).ToArray());
    Console.WriteLine($"{label} IQM={s.score:F6}, IC95%=[{s.low:F6},{s.high:F6}]");
}

Console.WriteLine($"Moyenne glissante : {Stats.Moving(baseline[0].Rewards).Length}points, épisodes300..12000 ; bootstrap2000réplications de20graines.");


20 graines : moyenne glissante et intervalle interquartile -30 -20 -10 0 10 300 3000 6000 9000 12000 <polygon points='80.000,184.200 80.076,184.323 80.152,184.268 80.228,183.648 80.304,183.627 80.380,184.207 80.456,184.275 80.532,184.370 80.609,183.709 80.685,184.275 80.761,184.275 80.837,184.289 80.913,184.145 80.989,184.118 81.065,184.139 81.141,183.770 81.217,184.091 81.293,183.961 81.369,183.825 81.445,183.668 81.521,183.927 81.597,183.061 81.674,183.075 81.750,183.252 81.826,182.516 81.902,182.502 81.978,182.789 82.054,182.775 82.130,182.945 82.206,182.216 82.282,182.455 82.358,182.584 82.434,182.550 82.510,181.964 82.586,181.657 82.662,181.173 82.738,181.848 82.815,181.834 82.891,182.059 82.967,182.148 83.043,182.005 83.119,181.650 83.195,181.711 83.271,181.541 83.347,181.670 83.423,181.466 83.499,181.411 83.575,181.473 83.651,181.895 83.727,182.093 83.803,181.609 83.879,181.616 83.956,181.125 84.032,181.377 84.108,181.500 84.184,181.650 84.260,181.745 84.336,181.609 84.412,182.073 84.488,182.073 84.564,182.686 84.640,182.632 84.716,182.625 84.792,182.625 84.868,182.884 84.944,183.034 85.021,183.620 85.097,183.641 85.173,183.641 85.249,183.907 85.325,183.995 85.401,184.009 85.477,184.445 85.553,183.580 85.629,184.370 85.705,184.657 85.781,184.582 85.857,184.234 85.933,183.873 86.009,183.811 86.085,183.750 86.162,183.423 86.238,183.777 86.314,183.027 86.390,182.911 86.466,182.373 86.542,182.291 86.618,182.270 86.694,182.345 86.770,182.489 86.846,182.516 86.922,183.055 86.998,183.395 87.074,183.375 87.150,183.320 87.226,183.095 87.303,182.277 87.379,182.952 87.455,182.666 87.531,182.673 87.607,183.150 87.683,182.427 87.759,182.086 87.835,182.059 87.911,182.093 87.987,181.984 88.063,181.916 88.139,182.673 88.215,182.461 88.291,181.698 88.368,181.991 88.444,182.080 88.520,182.318 88.596,182.441 88.672,181.357 88.748,182.052 88.824,181.561 88.900,181.745 88.976,181.991 89.052,181.957 89.128,181.841 89.204,181.984 89.280,181.902 89.356,181.827 89.432,181.473 89.509,181.234 89.585,181.091 89.661,181.214 89.737,180.948 89.813,180.948 89.889,181.023 89.965,180.955 90.041,181.773 90.117,182.073 90.193,182.161 90.269,182.468 90.345,182.673 90.421,183.395 90.497,183.266 90.574,183.293 90.650,183.470 90.726,183.136 90.802,183.457 90.878,183.130 90.954,183.170 91.030,183.709 91.106,183.075 91.182,182.195 91.258,182.161 91.334,182.120 91.410,182.039 91.486,182.080 91.562,181.998 91.638,182.223 91.715,182.223 91.791,182.202 91.867,181.377 91.943,181.943 92.019,182.039 92.095,182.168 92.171,181.868 92.247,182.223 92.323,182.100 92.399,182.134 92.475,181.732 92.551,180.770 92.627,180.893 92.703,180.041 92.779,180.443 92.856,180.273 92.932,180.477 93.008,180.341 93.084,180.198 93.160,180.198 93.236,179.802 93.312,179.373 93.388,179.230 93.464,179.216 93.540,179.080 93.616,178.991 93.692,178.575 93.768,178.480 93.844,178.950 93.921,179.264 93.997,179.141 94.073,179.298 94.149,179.120 94.225,179.175 94.301,178.882 94.377,178.882 94.453,178.827 94.529,178.739 94.605,178.480 94.681,178.582 94.757,178.425 94.833,178.282 94.909,178.030 94.985,178.030 95.062,177.968 95.138,177.968 95.214,178.009 95.290,177.334 95.366,177.075 95.442,177.109 95.518,177.102 95.594,176.509 95.670,176.557 95.746,176.530 95.822,176.468 95.898,176.195 95.974,176.809 96.050,176.782 96.126,176.652 96.203,176.564 96.279,176.577 96.355,176.802 96.431,176.039 96.507,175.780 96.583,175.432 96.659,175.555 96.735,175.445 96.811,175.411 96.887,175.043 96.963,174.784 97.039,174.334 97.115,174.368 97.191,175.002 97.268,175.084 97.344,175.166 97.420,175.330 97.496,175.445 97.572,175.493 97.648,175.643 97.724,175.541 97.800,175.425 97.876,175.486 97.952,175.343 98.028,174.593 98.104,174.457 98.180,174.614 98.256,174.566 98.332,175.282 98.409,175.234 98.485,175.500 98.561,175.166 98.637,174.450 98.713,174.307 98.789,174.416 98.865,174.211 98.941,173.932 99.017,174.648 99.093,175.111 99.169,175.111 99.245,174.450 99.321,174.477 99.397,174.273 99.474,174.102 99.550,174.143 99.626,173.5

IQL IQM=4.950700, IC95%=[4.915298,5.009918]


Hysteretic IQM=10.315800, IC95%=[10.226390,10.399515]


Moyenne glissante : 11701points, épisodes300..12000 ; bootstrap2000réplications de20graines.


### Interprétation des courbes et des intervalles

Les valeurs IQM et intervalles ci-dessus sont calculées sur les vingt retours terminaux C#. La bande colorée représente les quantiles intergraines des trajectoires lissées, pas un IC. Les RNG .NET et NumPy diffèrent ; comparer budgets, mécanismes et destinations, sans imposer les taux historiques Python. Une disjonction des IC décrit ce banc fini et ne démontre pas une garantie de convergence.

Mesures C# : IQL 4,950700, IC [4,915298 ; 5,009918] ; hystérésis 10,315800, IC [10,226390 ; 10,399515]. Les destinations initiales sont 19 C/C et 1 B/C pour IQL, 20 A/A pour l’hystérésis. Ces mesures seedées appartiennent à ce protocole.


<a id="exercice-2"></a>

### Exercice 2 — Changer l’unité statistique

Pourquoi un bootstrap des épisodes, au lieu des graines, produirait-il un intervalle artificiellement étroit ? **Indice** : les épisodes successifs ne sont pas indépendants.

Formulez votre réponse en termes d'unité d'échantillonnage. Deux épisodes adjacents partagent les mêmes tables Q et presque le même $\varepsilon$ ; les considérer indépendants créerait une pseudo-réplication. Une extension plus ambitieuse pourrait employer un bootstrap par blocs temporels pour étudier la dynamique, mais elle ne remplacerait toujours pas les graines indépendantes pour comparer les méthodes.


Partie calculable : `ReassemblerRuns(runs, indices)` retourne les runs complets rééchantillonnés dans l’ordre des indices, doublons inclus. Les fixtures ont des longueurs différentes et un tirage vide. Le harnais vérifie les blocs, puis compare moyenne par run et moyenne de tous les épisodes ; il ne certifie pas une justification textuelle.


In [8]:
#nullable enable
string? explicationDependance = null; // TODO étudiant : unité statistique, justification à discuter.
static double[][]? ReassemblerRuns(double[][] runs, int[] indices)
{
    // TODO étudiant : blocs complets, ordre et doublons.
    return null;
}

bool fini2 = true;
foreach (var(runs, indices)in new[]
{
    (new[] { new double[] { 1, 2 }, new double[] { 10, 20, 30 } }, new[] { 1, 1, 0 }),
    (new[] { new double[] { 3 }, new double[] { 4, 5 }, new double[] { 6, 7, 8, 9 } }, new[] { 2, 0 }),
    (new[] { new double[] { 1 }, new double[] { 2, 3 } }, Array.Empty<int>())
}

)
{
    var r = ReassemblerRuns(runs.Select(a => (double[])a.Clone()).ToArray(), (int[])indices.Clone());
    if (r is null)
    {
        fini2 = false;
        break;
    }

    Climbing.Check(r.Length == indices.Length, "nombre de blocs");
    for (int i = 0; i < r.Length; i++)
        Climbing.Check(r[i] is not null && r[i].All(double.IsFinite) && r[i].SequenceEqual(runs[indices[i]]), "run entier et ordre");
    if (r.Length > 0)
        Console.WriteLine($"Moyenne par run={r.Average(a => a.Average())}, moyenne épisodes={r.SelectMany(a => a).Average()}");
}

Console.WriteLine(!fini2 ? "Exercice 2 à compléter : runs complets" : explicationDependance is null ? "Blocs validés ; texte à compléter" : "Blocs validés ; explication reçue à discuter");


Exercice 2 à compléter : runs complets


## 7. Ablation du pessimisme

Nous comparons $\alpha^-\in\{0{,}1;0{,}05;0{,}02;0{,}01\}$. Les quatre valeurs sont préspécifiées et toutes recalculées, sans supprimer un réglage après observation.

### Lecture attendue

L'ablation teste une relation de dose plutôt qu'une opposition binaire. `0.10` reproduit exactement IQL ; `0.05` teste une correction négative seulement deux fois plus lente ; `0.02` sonde une zone intermédiaire où plusieurs destinations peuvent être observées ; `0.01` représente l'hysteresis forte. Chaque ligne est recalculée sur les mêmes vingt graines et rapporte à la fois la fréquence optimale et l'IQM avec son intervalle.

Cette grille évite deux erreurs de présentation : sélectionner après coup la seule valeur gagnante, ou conclure qu'il suffit d'imposer strictement $\alpha^-<\alpha^+$. Si `0.05` reste sous-optimal, l'inégalité seule n'est pas suffisante ; son amplitude est une composante empirique du remède.


In [9]:
var ablation = new Dictionary<double, (double rate, double score, double low, double high)>();
foreach (double an in new[]
{
    .1,
    .05,
    .02,
    .01
}

)
{
    var runs = Climbing.Seeds.Select(seed => Protocole.Run(seed, an)).ToArray();
    double rate = runs.Count(r => r.Dominant == 0) / (double)runs.Length;
    var s = Stats.Bootstrap(runs.Select(r => r.TailReturn).ToArray());
    ablation[an] = (rate, s.score, s.low, s.high);
    Console.WriteLine($"alpha-={an:F2} | (A,A)={rate:P0} | IQM={s.score:F6} | IC95%=[{s.low:F6},{s.high:F6}]");
    Protocole.Destinations($"Ablation{an}", runs);
}

Console.WriteLine($"Budgets effectivement exécutés : {Climbing.Runs}runs, {Climbing.Interactions}interactions.");
Climbing.Check(Climbing.Runs == 120 && Climbing.Interactions == 1440000, "120runs recalculés, budget complet");


alpha-=0.10 | (A,A)=0 % | IQM=4.950700 | IC95%=[4.915298,5.009918]


Ablation0.1 : (C,C):19, (B,C):1


Coordination suivie(A,A) : 0 %


Coordination suivie(B,B) : 0 %


Coordination suivie(C,C) : 95 %


Dominance moyenne dans les queues : 87.70 %


... (15 outputs left out)

Coordination suivie(B,B) : 0 %


Coordination suivie(C,C) : 0 %


Dominance moyenne dans les queues : 97.45 %


Budgets effectivement exécutés : 120runs, 1440000interactions.


### Interprétation de l’ablation

Lire les quatre fréquences et leurs IQM/IC calculés ci-dessus. Toutes les valeurs sont recalculées, y compris les deux taux déjà employés dans la comparaison initiale. Une transition ou un plateau sur cette grille ne définit pas un seuil mathématique. Les taux mesurés et destinations montrent si l’optimisme protège A dans cette exécution ; ils ne rendent pas l’hystérésis universelle. L’exercice suivant examine le cas extrême alpha−0.

Sur ces graines.NET, les quatre fréquences A/A sont0%,0%,65%,100%. À0,02,13graines ont A/A et7B/B ; la sortie Python donne une autre fréquence sur ses propres tirages.


<a id="exercice-3"></a>

### Exercice 3 — Tester l’autre extrême

Ajoutez $\alpha^-=0$. Un agent qui n’apprend jamais des erreurs négatives peut-il rester optimiste face à une action réellement mauvaise dans un autre jeu ? Proposez une matrice contre-exemple.

Votre contre-exemple doit séparer « mauvaise coordination temporaire » et « action intrinsèquement risquée ». Par exemple, construisez une action dont le gain rare est positif mais dont la plupart des résultats coordonnés restent négatifs. Précisez ensuite quelle statistique vous compareriez pour montrer que $\alpha^-=0$ conserve à tort cette action.


Partie calculable : `MatriceContreExemple()` fournit une matrice3×3 finie, avec |gain|≤1000. Pour isoler le risque sous un partenaire uniforme, sa première ligne contient un seul gain positif, deux négatifs et une moyenne strictement négative ; les autres lignes sont nulles (option sûre). Le harnais soumet réellement les gains de cette ligne en cycle à l’agent : alpha−0 conserve son maximum positif, tandis que la moyenne de Q sous alpha−0,1 suit le rendement négatif en régime périodique. Ce flux de diagnostic représente un partenaire imposé ; les simulations de deux agents adaptatifs sont ensuite mesurées sans imposer leur résultat. Si la matrice est fournie, vingt runs alpha−0 sur le jeu original, puis vingt runs par taux0et0,1 sur la matrice proposée complètent l’expérience.


In [10]:
#nullable enable
static double[, ]? MatriceContreExemple()
{
    // TODO étudiant : première ligne risquée, autres lignes nulles.
    return null;
}

var contreExemple = MatriceContreExemple();
if (contreExemple is null)
    Console.WriteLine("Exercice 3 à compléter : contre-exemple et alpha négatif zéro");
else
{
    Climbing.Matrix(contreExemple);
    var row = Enumerable.Range(0, 3).Select(j => contreExemple[0, j]).ToArray();
    Climbing.Check(contreExemple.Cast<double>().All(x => Math.Abs(x) <= 1000) && row.Count(x => x > 0) == 1 && row.Count(x => x < 0) == 2 && row.Average() < -1e-6 && Enumerable.Range(1, 2).All(i => Enumerable.Range(0, 3).All(j => contreExemple[i, j] == 0)), "ligne risquée et option sûre");
    var agents = new[]
    {
        new AgentTabulaire(.1, 0),
        new AgentTabulaire(.1, .1)
    };
    var traces = new[]
    {
        new List<double>(),
        new List<double>()
    };
    for (int t = 0; t < 3300; t++)
        for (int i = 0; i < 2; i++)
        {
            agents[i].Update(0, row[t % 3]);
            if (t >= 3000)
                traces[i].Add(agents[i].Q[0]);
        }

    Climbing.Check(traces[0].Min() > 0 && Math.Abs(traces[1].Average() - row.Average()) < 1e-8, "flux réellement observé alpha zéro/normal");
    Console.WriteLine($"Flux imposé : Q alpha-=0={traces[0].Average()}, Q alpha-=.1={traces[1].Average()}, rendement={row.Average()}");
    foreach (var(nom, m, an)in new[]
    {
        ("original alpha-=0", payoff, 0.0),
        ("contre-exemple alpha-=0", contreExemple, 0.0),
        ("contre-exemple alpha-=.1", contreExemple, .1)
    }

    )
    {
        var runs = Climbing.Seeds.Select(seed => Protocole.Run(seed, an, matrice: m)).ToArray();
        var s = Stats.Bootstrap(runs.Select(r => r.TailReturn).ToArray());
        Protocole.Destinations(nom, runs);
        Console.WriteLine($"IQM={s.score}, IC95%=[{s.low},{s.high}]");
    }
}


Exercice 3 à compléter : contre-exemple et alpha négatif zéro


## 8. Conclusion

| Question | Réponse |
|---|---|
| Pourquoi IQL se trompe-t-il ? | Les mauvaises coordinations contaminent une action conjointement optimale. |
| Que change l’hystérésis ? | Elle apprend plus lentement des TD-errors négatives, sans modifier la récompense. |
| Pourquoi plusieurs graines ? | Une trajectoire unique ne mesure pas la fréquence des destinations au budget fixé. |
| Pourquoi IQM + bootstrap ? | Pour ne pas dépendre de quelques graines extrêmes et quantifier l’incertitude. |
| Le remède est-il universel ? | Non : l’ablation conserve un réglage d’échec et l’optimisme extrême reste risqué. |

En MARL, une courbe de retour ne suffit pas : il faut mesurer **quelle paire terminale** domine chaque graine et relier ce résultat à la règle d’apprentissage.

## Limites et prolongements

- Le banc n'a qu'un état : il isole la sélection d’actions mais pas le crédit temporel.
- Les agents partagent une récompense et une architecture tabulaire ; les conclusions ne se transfèrent pas automatiquement aux jeux général-somme ni aux réseaux profonds.
- Vingt graines donnent une mesure honnête de la variabilité observée, pas une garantie sur toutes les initialisations.
- La classification par action conjointe dominante est adaptée à ce jeu discret. Un environnement continu demanderait un autre critère de convergence.

**Prolongement recommandé** : conserver le même budget et calendrier identiques et comparer hysteretic Q-learning à une exploration explicitement coordonnée. Si cette dernière récupère `(A,A)` sans asymétrie de taux, elle fournirait une intervention causale complémentaire sur l'autre maillon de la chaîne.

### Références

- Claus, C. et Boutilier, C. (1998), *The Dynamics of Reinforcement Learning in Cooperative Multiagent Systems*.
- Matignon, L., Laurent, G. J. et Le Fort-Piat, N. (2007), *Hysteretic Q-Learning: an Algorithm for Decentralized Reinforcement Learning in Cooperative Multi-Agent Teams*.
- Matignon, L., Laurent, G. J. et Le Fort-Piat, N. (2012), *Independent Reinforcement Learners in Cooperative Markov Games: a Survey Regarding Coordination Problems*.
- Hernandez-Leal, P., Kartal, B. et Taylor, M. E. (2019), *A Survey and Critique of Multiagent Deep Reinforcement Learning*.


## À retenir

- Le maximum conjoint est A/A, mais les retours sous exploration peuvent sous-évaluer une action optimale. C/C est une destination empirique possible, pas un Nash de cette matrice non symétrique.
- L’hystérésis réduit le poids des erreurs négatives. Elle peut protéger une bonne action ou conserver à tort une action à rendement négatif.
- Les taux de destinations, moyenne/IQR des trajectoires et IQM/bootstrap des graines mesurent des objets distincts. Les résultats de l’ablation restent des observations finies.
- Toutes les conditions gardent 20 graines × 12000 épisodes ; le même identifiant de graine ne certifie pas des aléas appariés entre politiques.

## Vérifiez votre compréhension

<details><summary>Une destination C/C est-elle un équilibre sous-optimal ?</summary>Non : l’agent de ligne gagne 6 en passant à B contre C, au lieu de 5. Les Nash purs sont A/A et B/B.</details>

<details><summary>Un TD-error négatif prouve-t-il une mauvaise coordination ?</summary>Non : il peut aussi venir d’une action réellement mauvaise ou d’un aléa. La matrice et les contrôles servent à distinguer ces explications.</details>

<details><summary>IQM et bootstrap remplacent-ils la moyenne ?</summary>Ils la complètent : la courbe montre la dynamique moyenne, l’IQR son hétérogénéité, et le bootstrap décrit l’incertitude de l’IQM entre graines. Il rééchantillonne des runs, pas des épisodes isolés.</details>

<details><summary>L’ablation établit-elle un seuil universel ?</summary>Non, elle évalue quatre taux sur cette matrice, avec vingt graines et un budget fini.</details>

## Exercices — récapitulatif

1. [Mise à jour et hypothèse](#exercice-1) : trajectoire Q numérique, justification à discuter.
2. [Unité statistique](#exercice-2) : blocs complets et réplications, justification à discuter.
3. [Optimisme extrême](#exercice-3) : matrice, flux discriminant et simulations conditionnelles.

## Pour aller plus loin

- [RL-07 — Apprentissage multi-agent](RL-07-Apprentissage-Multi-Agent-Python.ipynb).
- [RL-08 — Dyna-Q](RL-08-Dyna-Q-Planification-Python.ipynb).
- [GameTheory](../GameTheory/README.md) pour les correspondances de meilleures réponses.
- Comparer une exploration explicitement coordonnée à l’hystérésis, à budgets fixés, constitue une intervention complémentaire.
